# Product Promotion Inspector

This pipeline loads the raw data and filters for a specific `item_id` and `store_id` to visually and statistically inspect how different promotion types and values impact sales.

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

# Load the dataset
# As this notebook is in EDA/, the dataset is located one directory up
DATA_PATH = '../dataset/data_andre.feather'

print(f"Checking data path: {DATA_PATH}")
if os.path.exists(DATA_PATH):
    print(f"Loading data from {DATA_PATH}...")
    df = pd.read_feather(DATA_PATH)
    print(f"Dataset loaded with {len(df)} rows.")
else:
    print(f"Error: {DATA_PATH} not found. Please verify the path.")
    df = pd.DataFrame() # Fallback for code completion


Checking data path: ../dataset/data_andre.feather
Loading data from ../dataset/data_andre.feather...
Dataset loaded with 1082371 rows.


## 2. Set Target Product & Store
Define the parameters for the item and store you want to trace.

In [6]:
TARGET_ITEM_ID = 907969    # Replace with specific item_id
TARGET_STORE_ID = 6269    # Replace with specific store_id

# Filter specifically for this product and store combo
df_product = df[(df['item_id'] == TARGET_ITEM_ID) & (df['store_id'] == TARGET_STORE_ID)].copy()

# Sort Chronologically
if 'date' in df_product.columns:
    df_product['date'] = pd.to_datetime(df_product['date'])
    df_product = df_product.sort_values('date').reset_index(drop=True)

print(f"Filtered down to {len(df_product)} occurrences for Item: {TARGET_ITEM_ID} | Store: {TARGET_STORE_ID}")
df_product.head()

Filtered down to 761 occurrences for Item: 907969 | Store: 6269


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_value_DISC,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id
0,2021-01-23,907969,161,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
1,2021-01-24,907969,150,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
2,2021-01-25,907969,153,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
3,2021-01-26,907969,69,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
4,2021-01-27,907969,88,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269


## 3. Analyze Promotions
Identify which of the 8 promotion types were ever active for this product across its whole timeline. Then, compare sales statistics (Promo Days vs. Regular Days).

In [7]:
# Define all possible promotions to inspect
promo_types = [
    "promo_type_FRPG", "promo_type_GAS", "promo_type_BOGO", "promo_type_DISC",
    "promo_type_CIRC", "promo_type_CIRE", "promo_type_CLCP", "promo_type_LFPE"
]
promo_values = [
    "promo_value_FRPG", "promo_value_GAS", "promo_value_BOGO", "promo_value_DISC",
    "promo_value_CIRC", "promo_value_CIRE", "promo_value_CLCP", "promo_value_LFPE"
]

# Discover which promos are actually turned "on" at least once
active_promos = [p for p in promo_types if p in df_product.columns and df_product[p].sum() > 0]
print(f"Active promotion types for this product: {active_promos}\n")

if len(active_promos) > 0 and len(df_product) > 0:
    # Build a combined boolean mask marking if ANY promo is firing 
    df_product['any_promo_active'] = df_product[active_promos].max(axis=1)

    # Let's see some basic descriptive stats on sales (value) during non-promo vs promo
    print("Sales Summary Breakdown (0: No Promotion Active, 1: Any Promotion Active)")
    promo_stats = df_product.groupby('any_promo_active')['value'].agg(['mean', 'median', 'min', 'max', 'count', 'sum']).round(2)
    display(promo_stats)
    
    # We can also see sales broken down specifically by the ACTIVE promo types
    print("\nTotal Sum of Sales on days a specific promo type fired:")
    for promo in active_promos:
        promo_days_sales = df_product[df_product[promo] == 1]['value'].sum()
        print(f" - {promo.replace('promo_type_', '')}: {promo_days_sales} total units sold")
else:
    print("No promotions were found or the dataset is empty for this item/store combo.")

Active promotion types for this product: ['promo_type_CIRE']

Sales Summary Breakdown (0: No Promotion Active, 1: Any Promotion Active)


,mean,median,min,max,count,sum
any_promo_active,,,,,,
0,114.19,105.0,0,340,740,84497
1,169.76,176.0,31,287,21,3565



Total Sum of Sales on days a specific promo type fired:
 - CIRE: 3565 total units sold


## 4. Visualizing Promotions Over the Sales Timeline

Generate an interactive Plotly timeline of the item's sales overlaid with promotion indicators.

In [8]:
if not df_product.empty:
    # Initialize the plot 
    fig = go.Figure()

    # Base Trace: The continuous sales timeline
    fig.add_trace(go.Scatter(
        x=df_product['date'], 
        y=df_product['value'],
        mode='lines',
        name='Sales Value',
        line=dict(color='rgba(0, 102, 204, 0.7)', width=1.5),
        hovertemplate='Date: %{x}<br>Sales: %{y}'
    ))

    # Add dynamic traces for each ACTIVE promo 
    # Use different colors and distinct markers for each variant 
    colors = px.colors.qualitative.Set1
    
    for i, promo_col in enumerate(active_promos):
        # Subset rows where THIS promo type is precisely triggered (1)
        promo_days = df_product[df_product[promo_col] > 0]
        
        # Link to the value scale mapping to see what value is associated
        val_col = promo_col.replace('type', 'value')
        
        # Combine hovering info dynamically
        if val_col in df_product.columns:
            hover_text = [
                f"Sales: {s} <br>Promo Type: {promo_col.replace('promo_type_', '')}<br>Promo Val: {v}" 
                for v, s in zip(promo_days[val_col], promo_days['value'])
            ]
        else:
            hover_text = [
                f"Sales: {s} <br>Promo Type: {promo_col.replace('promo_type_', '')}" 
                for s in promo_days['value']
            ]
            
        fig.add_trace(go.Scatter(
            x=promo_days['date'], 
            y=promo_days['value'],
            mode='markers',
            name=f"{promo_col.replace('promo_type_', '')} Active",
            text=hover_text,
            hovertemplate='%{x|%b %d, %Y}<br>%{text}',
            marker=dict(
                size=11, 
                symbol='star', 
                color=colors[i % len(colors)], 
                line=dict(width=1, color='Black')
            )
        ))

    # Optional: Highlight specific splits, e.g. tests vs validation start 
    # if you want to visually see test ranges
    
    fig.update_layout(
        title=f'Sales vs Promotional Activity Timeline<br><sup>Item: {TARGET_ITEM_ID} | Store: {TARGET_STORE_ID}</sup>',
        xaxis_title='Date',
        yaxis_title='Sales (Units)',
        hovermode="x unified",
        template="plotly_white",
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig.show()